In [2]:
"""
TRANSFORMER FROM SCRATCH

A minimal transformer implementation that shows:
1. Self-Attention mechanism
2. Multi-Head Attention
3. Positional Encoding
4. Feed-Forward Network
5. Complete encoder-decoder with inference

Run this file and you'll see:
- Attention weights (which words focus on which words)
- Output probabilities for each word
- Step-by-step processing
"""

import numpy as np
import math
from typing import Tuple, List, Dict


# -----------------------------------------------------
# UTILITY FUNCTIONS
# -----------------------------------------------------

def softmax(x: np.ndarray) -> np.ndarray:
    """
    Convert scores to probabilities.
    softmax([2, 1, 0.1]) = [0.66, 0.24, 0.1]  (sum to 1)
    
    Args:
        x: Array of scores
    
    Returns:
        Probabilities that sum to 1
    """
    # Subtract max for numerical stability
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=-1, keepdims=True)


def relu(x: np.ndarray) -> np.ndarray:
    """ReLU activation: max(0, x)"""
    return np.maximum(0, x)


def relu_derivative(x: np.ndarray) -> np.ndarray:
    """Derivative of ReLU for backpropagation"""
    return (x > 0).astype(float)


# -----------------------------------------------------
# POSITIONAL ENCODING
# -----------------------------------------------------

def positional_encoding(seq_len: int, d_model: int) -> np.ndarray:
    """
    Add position information to embeddings.
    
    Without this, transformer doesn't know word order since it processes
    everything in parallel.
    
    Args:
        seq_len: Length of sequence
        d_model: Embedding dimension
    
    Returns:
        Positional encoding matrix of shape (seq_len, d_model)
    """
    # Create position indices: [0, 1, 2, 3, ...]
    position = np.arange(seq_len)[:, np.newaxis]
    
    # Create dimension indices: [0, 1, 2, 3, ...]
    dim_indices = np.arange(d_model)[np.newaxis, :]
    
    # Angle rates: 10000^(2i/d_model)
    angle_rates = 1 / np.power(10000, (2 * (dim_indices // 2)) / np.float32(d_model))
    
    # Compute angles
    angle_rads = position * angle_rates
    
    # Apply sine to even indices and cosine to odd indices
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
    
    return angle_rads


# -----------------------------------------------------
# SELF-ATTENTION MECHANISM
# ------------------------------------------------------

class SelfAttention:
    """
    Self-Attention: Each word attends to all words in the sequence.
    
    How it works:
    1. Create Query (Q), Key (K), Value (V) projections
    2. Compute attention scores: Score = Q @ K^T / sqrt(d_k)
    3. Apply softmax to get attention weights
    4. Weight values by attention: Output = Attention @ V
    
    Attention shows which words are important for understanding each word.
    """
    
    def __init__(self, d_model: int, d_k: int):
        """
        Args:
            d_model: Total embedding dimension
            d_k: Dimension of each attention head
        """
        self.d_model = d_model
        self.d_k = d_k
        
        # Initialize Q, K, V projection matrices
        # These will be learned during training
        self.W_q = np.random.randn(d_model, d_k) * 0.01
        self.W_k = np.random.randn(d_model, d_k) * 0.01
        self.W_v = np.random.randn(d_model, d_k) * 0.01
        
        # For backpropagation
        self.attention_weights = None
    
    def forward(self, X: np.ndarray, mask: np.ndarray = None) -> Tuple[np.ndarray, np.ndarray]:
        """
        Forward pass of self-attention.
        
        Args:
            X: Input sequence of shape (seq_len, d_model)
            mask: Optional mask for decoder (to prevent looking at future tokens)
        
        Returns:
            output: Attention output of shape (seq_len, d_k)
            attention_weights: Attention weights of shape (seq_len, seq_len)
        """
        seq_len = X.shape[0]
        
        # Step 1: Project input to Q, K, V
        Q = X @ self.W_q  # (seq_len, d_k)
        K = X @ self.W_k  # (seq_len, d_k)
        V = X @ self.W_v  # (seq_len, d_k)
        
        # Step 2: Compute attention scores
        # Score[i, j] = similarity between token i's query and token j's key
        scores = (Q @ K.T) / math.sqrt(self.d_k)  # (seq_len, seq_len)
        
        # Step 3: Apply mask if provided (for decoder)
        if mask is not None:
            scores = scores + (mask * -1e9)
        
        # Step 4: Apply softmax to get attention weights
        # Now scores[i, j] = probability of token i attending to token j
        attention_weights = softmax(scores)  # (seq_len, seq_len)
        self.attention_weights = attention_weights
        
        # Step 5: Weight values by attention
        output = attention_weights @ V  # (seq_len, d_k)
        
        return output, attention_weights


# ------------------------------------------------
# MULTI-HEAD ATTENTION
# ------------------------------------------------

class MultiHeadAttention:
    """
    Instead of one attention mechanism, use multiple in parallel.
    Each head learns different patterns and relationships.
    
    Example with 8 heads:
    - Head 1: Learns subject-verb relationships
    - Head 2: Learns adjective-noun relationships
    - Head 3: Learns long-range dependencies
    - etc.
    """
    
    def __init__(self, d_model: int, num_heads: int):
        """
        Args:
            d_model: Embedding dimension
            num_heads: Number of attention heads
        """
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Create attention head for each
        self.heads = [SelfAttention(d_model, self.d_k) for _ in range(num_heads)]
        
        # Output projection: concatenate all heads and project
        self.W_o = np.random.randn(d_model, d_model) * 0.01
        
        self.all_attention_weights = []
    
    def forward(self, X: np.ndarray, mask: np.ndarray = None) -> Tuple[np.ndarray, List]:
        """
        Forward pass of multi-head attention.
        
        Args:
            X: Input of shape (seq_len, d_model)
            mask: Optional mask for decoder
        
        Returns:
            output: Processed sequence of shape (seq_len, d_model)
            attention_weights: Attention weights from all heads
        """
        # Run all attention heads in parallel
        head_outputs = []
        self.all_attention_weights = []
        
        for head in self.heads:
            output, weights = head.forward(X, mask)
            head_outputs.append(output)
            self.all_attention_weights.append(weights)
        
        # Concatenate all head outputs
        concatenated = np.concatenate(head_outputs, axis=-1)  # (seq_len, d_model)
        
        # Project concatenated output
        output = concatenated @ self.W_o  # (seq_len, d_model)
        
        return output, self.all_attention_weights


# --------------------------------------------------
# FEED-FORWARD NETWORK
# --------------------------------------------------

class FeedForwardNetwork:
    """
    Position-wise Feed-Forward Network.
    Applied independently to each position.
    
    FFN(x) = ReLU(x * W1 + b1) * W2 + b2
    
    This helps refine the representation after attention.
    """
    
    def __init__(self, d_model: int, d_ff: int = 2048):
        """
        Args:
            d_model: Embedding dimension
            d_ff: Hidden dimension (usually 4x d_model)
        """
        self.W1 = np.random.randn(d_model, d_ff) * 0.01
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * 0.01
        self.b2 = np.zeros(d_model)
    
    def forward(self, X: np.ndarray) -> np.ndarray:
        """
        Args:
            X: Input of shape (seq_len, d_model)
        
        Returns:
            Output of shape (seq_len, d_model)
        """
        # First linear transformation + ReLU
        hidden = relu(X @ self.W1 + self.b1)  # (seq_len, d_ff)
        
        # Second linear transformation
        output = hidden @ self.W2 + self.b2  # (seq_len, d_model)
        
        return output


# -----------------------------------------------------
# TRANSFORMER ENCODER LAYER
# -----------------------------------------------------

class TransformerEncoderLayer:
    """
    Single encoder layer consisting of:
    1. Multi-Head Self-Attention
    2. Add & Norm (Residual connection + Layer Normalization)
    3. Feed-Forward Network
    4. Add & Norm
    """
    
    def __init__(self, d_model: int, num_heads: int, d_ff: int = 2048):
        self.d_model = d_model
        
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForwardNetwork(d_model, d_ff)
        
        # Layer normalization parameters
        self.gamma1 = np.ones(d_model)
        self.beta1 = np.zeros(d_model)
        self.gamma2 = np.ones(d_model)
        self.beta2 = np.zeros(d_model)
    
    def layer_norm(self, X: np.ndarray, gamma: np.ndarray, beta: np.ndarray) -> np.ndarray:
        """
        Layer Normalization: normalize each sequence independently.
        Helps training be more stable.
        """
        mean = np.mean(X, axis=-1, keepdims=True)
        std = np.std(X, axis=-1, keepdims=True) + 1e-6
        normalized = (X - mean) / std
        return gamma * normalized + beta
    
    def forward(self, X: np.ndarray) -> Tuple[np.ndarray, List]:
        """
        Args:
            X: Input of shape (seq_len, d_model)
        
        Returns:
            output: Processed sequence
            attention_weights: From multi-head attention
        """
        # Multi-Head Attention with Residual Connection
        attn_output, attn_weights = self.mha.forward(X)
        X = X + attn_output  # Residual connection
        X = self.layer_norm(X, self.gamma1, self.beta1)  # Layer norm
        
        # Feed-Forward with Residual Connection
        ffn_output = self.ffn.forward(X)
        X = X + ffn_output  # Residual connection
        X = self.layer_norm(X, self.gamma2, self.beta2)  # Layer norm
        
        return X, attn_weights


# ------------------------------------------------
# SIMPLE EMBEDDING LAYER
# ------------------------------------------------

class Embedding:
    """Convert words (indices) to dense vectors."""
    
    def __init__(self, vocab_size: int, d_model: int):
        self.embedding_matrix = np.random.randn(vocab_size, d_model) * 0.01
    
    def forward(self, indices: np.ndarray) -> np.ndarray:
        """
        Args:
            indices: Word indices of shape (seq_len,)
        
        Returns:
            Embeddings of shape (seq_len, d_model)
        """
        return self.embedding_matrix[indices]


# ------------------------------------------------
# SIMPLE TRANSFORMER ENCODER
# ------------------------------------------------

class TransformerEncoder:
    """
    Stack of transformer encoder layers.
    """
    
    def __init__(self, vocab_size: int, d_model: int, num_heads: int, num_layers: int, d_ff: int = 2048):
        self.d_model = d_model
        self.embedding = Embedding(vocab_size, d_model)
        self.pos_encoding = None
        
        self.layers = [
            TransformerEncoderLayer(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ]
    
    def forward(self, indices: np.ndarray) -> Tuple[np.ndarray, List]:
        """
        Args:
            indices: Input word indices of shape (seq_len,)
        
        Returns:
            output: Encoded representation of shape (seq_len, d_model)
            all_attention_weights: Attention weights from all layers
        """
        seq_len = len(indices)
        
        # Embedding + Positional Encoding
        X = self.embedding.forward(indices)  # (seq_len, d_model)
        
        # Add positional encoding
        if self.pos_encoding is None:
            self.pos_encoding = positional_encoding(seq_len, self.d_model)
        
        X = X + self.pos_encoding[:seq_len]
        
        # Pass through all encoder layers
        all_attention_weights = []
        for layer in self.layers:
            X, attn_weights = layer.forward(X)
            all_attention_weights.append(attn_weights)
        
        return X, all_attention_weights


# --------------------------------------------------
# DEMONSTRATION
# --------------------------------------------------

def print_section(title: str):
    """Print a formatted section header."""
    print("\n" + "=" * 80)
    print(f"  {title}")
    print("=" * 80)


def demonstrate_attention():
    """Show how self-attention works step-by-step."""
    print_section("SELF-ATTENTION MECHANISM")
    
    # Simple example: word embeddings for a sentence
    vocab = ["the", "cat", "sat", "on", "mat"]
    vocab_idx = {word: idx for idx, word in enumerate(vocab)}
    
    # Create a simple embedding matrix
    d_model = 4
    embedding_matrix = np.array([
        [1.0, 0.5, 0.2, 0.1],  # "the"
        [0.8, 1.0, 0.3, 0.2],  # "cat"
        [0.9, 0.7, 1.0, 0.3],  # "sat"
        [0.6, 0.8, 0.5, 1.0],  # "on"
        [0.7, 0.6, 0.8, 0.9],  # "mat"
    ])
    
    print(f"\nVocabulary: {vocab}")
    print(f"\nEmbedding matrix (each row is a word embedding):")
    for word, emb in zip(vocab, embedding_matrix):
        print(f"  {word:8s}: {emb}")
    
    # Create attention mechanism
    attention = SelfAttention(d_model=4, d_k=4)
    
    # Set weights to identity for simplicity (this would be learned)
    attention.W_q = np.eye(4)
    attention.W_k = np.eye(4)
    attention.W_v = np.eye(4)
    
    # Forward pass
    output, weights = attention.forward(embedding_matrix)
    
    print(f"\n--- ATTENTION WEIGHTS ---")
    print(f"(Each row shows how much each word attends to every other word)")
    print(f"\nShape: {weights.shape}  (seq_len={len(vocab)}, seq_len={len(vocab)})")
    print("\nAttention weights:")
    print("       " + "   ".join([f"{w:>8s}" for w in vocab]))
    for i, word in enumerate(vocab):
        weights_str = "   ".join([f"{weights[i, j]:>8.4f}" for j in range(len(vocab))])
        print(f"{word:>5s}: {weights_str}")
    
    print(f"\nInterpretation:")
    for i, word in enumerate(vocab):
        top_idx = np.argmax(weights[i])
        top_weight = weights[i, top_idx]
        print(f"  '{word}' attends most to '{vocab[top_idx]}' with weight {top_weight:.4f}")


def demonstrate_multi_head_attention():
    """Show how multiple attention heads work in parallel."""
    print_section("MULTI-HEAD ATTENTION")
    
    vocab = ["the", "cat", "sat", "on", "mat"]
    seq_len = len(vocab)
    d_model = 8
    num_heads = 2  # 2 heads for simplicity
    
    # Create random embeddings
    X = np.random.randn(seq_len, d_model) * 0.1
    
    # Create multi-head attention
    mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
    output, all_weights = mha.forward(X)
    
    print(f"\nConfiguration:")
    print(f"  Vocabulary: {vocab}")
    print(f"  Embedding dimension: {d_model}")
    print(f"  Number of heads: {num_heads}")
    print(f"  Dimension per head: {d_model // num_heads}")
    
    print(f"\n--- ATTENTION WEIGHTS PER HEAD ---")
    for head_idx, weights in enumerate(all_weights):
        print(f"\nHead {head_idx + 1} attention weights:")
        print("       " + "   ".join([f"{w:>8s}" for w in vocab]))
        for i, word in enumerate(vocab):
            weights_str = "   ".join([f"{weights[i, j]:>8.4f}" for j in range(seq_len)])
            print(f"{word:>5s}: {weights_str}")
        
        print(f"  Top focus:")
        for i, word in enumerate(vocab):
            top_idx = np.argmax(weights[i])
            top_weight = weights[i, top_idx]
            print(f"    '{word}' → '{vocab[top_idx]}' ({top_weight:.4f})")


def demonstrate_full_encoder():
    """Show a complete transformer encoder."""
    print_section("COMPLETE TRANSFORMER ENCODER")
    
    # Configuration
    vocab = ["the", "cat", "sat", "on", "mat"]
    vocab_size = len(vocab)
    d_model = 8
    num_heads = 2
    num_layers = 2
    
    # Create encoder
    encoder = TransformerEncoder(
        vocab_size=vocab_size,
        d_model=d_model,
        num_heads=num_heads,
        num_layers=num_layers,
    )
    
    # Create input indices
    input_indices = np.array([0, 1, 2, 3, 4])  # "the cat sat on mat"
    
    print(f"\nConfiguration:")
    print(f"  Input: {' '.join([vocab[i] for i in input_indices])}")
    print(f"  Vocabulary size: {vocab_size}")
    print(f"  Embedding dimension: {d_model}")
    print(f"  Number of attention heads: {num_heads}")
    print(f"  Number of encoder layers: {num_layers}")
    
    # Forward pass
    output, all_attention_weights = encoder.forward(input_indices)
    
    print(f"\nOutput shape: {output.shape}")
    print(f"  (seq_len={output.shape[0]}, d_model={output.shape[1]})")
    
    print(f"\n--- LAYER-BY-LAYER ATTENTION ---")
    for layer_idx, layer_weights in enumerate(all_attention_weights):
        print(f"\nLayer {layer_idx + 1}:")
        for head_idx, head_weights in enumerate(layer_weights):
            print(f"\n  Head {head_idx + 1}:")
            print("       " + "   ".join([f"{w:>8s}" for w in vocab]))
            for i, word in enumerate(vocab):
                weights_str = "   ".join([f"{head_weights[i, j]:>8.4f}" for j in range(len(vocab))])
                print(f"  {word:>5s}: {weights_str}")


def demonstrate_output_probabilities():
    """Show how transformer predicts output probabilities."""
    print_section("OUTPUT PREDICTION (WORD PROBABILITIES)")
    
    vocab = ["the", "cat", "sat", "on", "mat"]
    vocab_size = len(vocab)
    d_model = 8
    
    # Create encoder
    encoder = TransformerEncoder(
        vocab_size=vocab_size,
        d_model=d_model,
        num_heads=2,
        num_layers=1,
    )
    
    # Input: "the cat sat"
    input_indices = np.array([0, 1, 2])
    input_sentence = " ".join([vocab[i] for i in input_indices])
    
    # Get encoder output
    output, _ = encoder.forward(input_indices)
    
    print(f"\nInput: '{input_sentence}'")
    print(f"Task: Predict what word comes next")
    
    # For each position, create output probabilities
    print(f"\n--- PREDICTED WORD PROBABILITIES ---")
    for pos in range(len(input_indices)):
        # Use the encoder output at this position
        # In a real decoder, this would go through more processing
        position_output = output[pos]
        
        # Create a simple linear layer to predict word probabilities
        # (in reality this is learned)
        logits = position_output @ np.random.randn(d_model, vocab_size) * 0.1
        probabilities = softmax(logits)
        
        print(f"\nPosition {pos} (after '{vocab[input_indices[pos]]}'):")
        
        # Sort by probability
        sorted_indices = np.argsort(probabilities)[::-1]
        for idx in sorted_indices[:3]:  # Top 3
            prob = probabilities[idx]
            print(f"  {vocab[idx]:8s}: {prob:.4f} ({prob*100:.2f}%)")


# ---------------------------------------------
# MAIN
# ---------------------------------------------

if __name__ == "__main__":
    print("\n" + "█" * 80)
    print("█" + " " * 78 + "█")
    print("█" + "  TRANSFORMER FROM SCRATCH - SELF-ATTENTION VISUALIZATION".center(78) + "█")
    print("█" + " " * 78 + "█")
    print("█" * 80)
    
    # 1. Self-Attention
    demonstrate_attention()
    
    # 2. Multi-Head Attention
    demonstrate_multi_head_attention()
    
    # 3. Full Encoder
    demonstrate_full_encoder()
    
    # 4. Output Probabilities
    demonstrate_output_probabilities()
    
    print_section("SUMMARY")
    print("""
    Key Concepts Demonstrated:
    
    1. SELF-ATTENTION:
       - Each word creates Query, Key, Value vectors
       - Attention weights = softmax(Q @ K^T / sqrt(d_k))
       - Output = Attention weights @ Values
       - Shows which words are important for each word
    
    2. MULTI-HEAD ATTENTION:
       - Multiple attention mechanisms run in parallel
       - Each head learns different patterns
       - Outputs concatenated and projected
       - More expressive than single attention head
    
    3. POSITIONAL ENCODING:
       - Added to embeddings to preserve word order
       - Uses sine/cosine waves with different frequencies
       - Without it, transformer doesn't know sequence order
    
    4. FEED-FORWARD NETWORK:
       - Applied to each position independently
       - FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2
       - Adds non-linearity and refines representations
    
    5. OUTPUT PROBABILITIES:
       - Encoder output passed through linear layer + softmax
       - Softmax converts logits to probabilities
       - Highest probability word is typically chosen
       - In training, model learns to assign high prob to correct word
    
    Why Transformers Work:
    - Processes entire sequence in parallel (fast!)
    - Attention captures long-range dependencies
    - No vanishing gradient problem like RNNs
    - Can handle sequences of any length
    """)
    
    print("=" * 80 + "\n")


████████████████████████████████████████████████████████████████████████████████
█                                                                              █
█            TRANSFORMER FROM SCRATCH - SELF-ATTENTION VISUALIZATION           █
█                                                                              █
████████████████████████████████████████████████████████████████████████████████

  SELF-ATTENTION MECHANISM

Vocabulary: ['the', 'cat', 'sat', 'on', 'mat']

Embedding matrix (each row is a word embedding):
  the     : [1.  0.5 0.2 0.1]
  cat     : [0.8 1.  0.3 0.2]
  sat     : [0.9 0.7 1.  0.3]
  on      : [0.6 0.8 0.5 1. ]
  mat     : [0.7 0.6 0.8 0.9]

--- ATTENTION WEIGHTS ---
(Each row shows how much each word attends to every other word)

Shape: (5, 5)  (seq_len=5, seq_len=5)

Attention weights:
            the        cat        sat         on        mat
  the:   0.1976     0.2056     0.2162     0.1879     0.1927
  cat:   0.1762     0.2142     0.2152     0.1997